# NB06 — Markov Kernel of Hierarchy States on Real Data

Tests ChatGPT's "Formalization 2: Markov Kernel Hierarchy" on real OHLCV.

The target is no longer next-week price direction (which NB05 showed the hierarchy doesn't help with). It is **next-week hierarchy state**. Each row is assigned a discrete state `z_t ∈ {0..K-1}` based on which leaf contains its (greed_index, fear_index), and we ask: given the current state and current price features, can we predict `z_{t+5}`?

`K = 4` (`MAX_HIERARCHY_DEPTH = 2`). Random baseline = 25%. The hard baseline is **persistence**: predict `z_{t+5} = z_t`. Hierarchy states autocorrelate strongly on daily bars, so persistence is non-trivial.

Models compared:

1. **random** — uniform `1/K`.
2. **marginal** — always predict `argmax_z P(z)` on the training set.
3. **persistence** — `ẑ_{t+5} = z_t`.
4. **empirical Markov kernel** — `ẑ_{t+5} = argmax_u P(z_{t+5}=u | z_t)` estimated from training counts. Uses only `z_t`.
5. **features-only RF** — `ẑ_{t+5} = RF(baseline_features)`. Uses only price/volume context, no `z_t` at all.
6. **joint RF** — `ẑ_{t+5} = RF(baseline_features + soft_membership_t)`. The full Markov kernel: feature context + state representation.

The interesting comparisons are *joint RF vs persistence* (does the model beat the trivial autocorrelation?) and *joint RF vs features-only RF* (does adding `z_t` information add anything beyond features?).

Each run appends an entry to `EXPERIMENTS.md`.


In [1]:
import math
import os
import json
import random
from dataclasses import dataclass
from datetime import datetime
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TICKERS = ["SPY", "NVDA", "BTC-USD"]
START_DATE = "2010-01-01"
END_DATE = None
HORIZON = 5
TEST_FRACTION = 0.20
VALIDATION_FRACTION = 0.15

MAX_HIERARCHY_DEPTH = 2
MIN_LEAF_COUNT = 200

RF_N_ESTIMATORS = 300
RF_GRID = [
    {"max_depth": md_, "min_samples_leaf": msl}
    for md_ in (4, 6, 8)
    for msl in (10, 20, 40)
]

JOURNAL_PATH = "EXPERIMENTS.md"


## Reused machinery from NB04/NB05

Data loading, feature construction, hierarchy build, and time split are unchanged from NB05.


In [2]:
def load_yfinance_ohlcv(ticker: str, start_date: str, end_date: Optional[str] = None) -> pd.DataFrame:
    raw = yf.download(
        tickers=ticker, start=start_date, end=end_date,
        auto_adjust=True, progress=False, group_by="column",
    )
    if raw.empty:
        raise RuntimeError(f"No data returned for ticker: {ticker}")
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = [c[0].lower() for c in raw.columns]
    else:
        raw.columns = [str(c).lower() for c in raw.columns]
    required = ["open", "high", "low", "close", "volume"]
    df = raw[required].copy().reset_index()
    df = df.rename(columns={"Date": "date", "Datetime": "date"})
    df["date"] = pd.to_datetime(df["date"])
    return df.dropna().reset_index(drop=True)


def rolling_zscore(x, window, eps=1e-8):
    mean = x.rolling(window, min_periods=max(3, window // 3)).mean()
    std = x.rolling(window, min_periods=max(3, window // 3)).std()
    return (x - mean) / (std + eps)


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))


def make_real_market_features(df, horizon):
    out = df.copy()
    for lag in (1, 2, 5, 10, 20):
        out[f"ret_{lag}"] = out["close"].pct_change(lag)
    out["range_pct"] = (out["high"] - out["low"]) / out["close"]
    out["open_close_pct"] = (out["close"] - out["open"]) / out["open"]
    for w in (5, 10, 20):
        out[f"vol_{w}"] = out["ret_1"].rolling(w).std()
    for w in (10, 20, 50):
        out[f"ma_{w}"] = out["close"].rolling(w).mean()
        out[f"price_to_ma_{w}"] = out["close"] / out[f"ma_{w}"] - 1.0
    out["volume_log"] = np.log1p(out["volume"])
    out["volume_z_20"] = rolling_zscore(out["volume_log"], 20)
    for w in (20, 50):
        out[f"rolling_max_{w}"] = out["close"].rolling(w).max()
        out[f"drawdown_{w}"] = out["close"] / out[f"rolling_max_{w}"] - 1.0
    out["trend_pressure"] = rolling_zscore(out["ret_10"], 30)
    out["fear_pressure"] = (
        -4.0 * out["drawdown_20"] - 2.0 * out["drawdown_50"]
        + 8.0 * out["vol_10"] + 0.35 * out["volume_z_20"]
        - 0.50 * out["ret_5"]
    )
    out["greed_pressure"] = (
        1.50 * out["trend_pressure"] + 2.00 * out["price_to_ma_20"]
        + 1.00 * out["ret_20"] + 0.20 * out["volume_z_20"]
        - 3.00 * out["vol_20"]
    )
    out["fear_index"] = sigmoid_np(out["fear_pressure"].clip(-8, 8))
    out["greed_index"] = sigmoid_np(out["greed_pressure"].clip(-8, 8))
    out["future_close"] = out["close"].shift(-horizon)
    out["future_return"] = out["future_close"] / out["close"] - 1.0
    out["future_direction"] = (out["future_return"] > 0).astype(int)
    out = out.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    return out


BASELINE_FEATURES = [
    "ret_1", "ret_2", "ret_5", "ret_10", "ret_20",
    "range_pct", "open_close_pct",
    "vol_5", "vol_10", "vol_20",
    "price_to_ma_10", "price_to_ma_20", "price_to_ma_50",
    "volume_z_20", "drawdown_20", "drawdown_50",
]


@dataclass
class MeasureNode:
    node_id: str
    depth: int
    bounds: Dict[str, Tuple[float, float]]
    mass: float
    count: int
    mean: Dict[str, float]
    children: Optional[List["MeasureNode"]] = None

    @property
    def is_leaf(self):
        return not self.children


def in_bounds(row, bounds):
    for col, (lo, hi) in bounds.items():
        v = float(row[col])
        if not (lo <= v < hi):
            return False
    return True


def split_node(node, split_col, split_value):
    lo, hi = node.bounds[split_col]
    lb, rb = dict(node.bounds), dict(node.bounds)
    lb[split_col] = (lo, split_value)
    rb[split_col] = (split_value, hi)
    node.children = [
        MeasureNode(node.node_id + "L", node.depth + 1, lb, node.mass / 2.0, 0, {}, None),
        MeasureNode(node.node_id + "R", node.depth + 1, rb, node.mass / 2.0, 0, {}, None),
    ]
    return node.children


def collect_leaves(node):
    if node.is_leaf:
        return [node]
    leaves = []
    for c in node.children:
        leaves.extend(collect_leaves(c))
    return leaves


def build_hierarchy(train_df, max_depth=2, min_count=200):
    feature_cols = ["greed_index", "fear_index"]
    root = MeasureNode(
        "root", 0,
        {"greed_index": (0.0, 1.000001), "fear_index": (0.0, 1.000001)},
        1.0, len(train_df),
        {c: float(train_df[c].mean()) for c in feature_cols}, None,
    )
    queue = [root]
    while queue:
        node = queue.pop(0)
        if node.depth >= max_depth:
            continue
        mask = train_df.apply(lambda r: in_bounds(r, node.bounds), axis=1)
        subset = train_df[mask]
        node.count = len(subset)
        node.mass = node.count / max(len(train_df), 1)
        if node.count < min_count:
            continue
        variances = {c: float(subset[c].var()) for c in feature_cols}
        split_col = max(variances, key=variances.get)
        split_value = float(subset[split_col].median())
        if node.bounds[split_col][0] < split_value < node.bounds[split_col][1]:
            queue.extend(split_node(node, split_col, split_value))
    for leaf in collect_leaves(root):
        mask = train_df.apply(lambda r: in_bounds(r, leaf.bounds), axis=1)
        subset = train_df[mask]
        leaf.count = len(subset)
        leaf.mass = leaf.count / max(len(train_df), 1)
        leaf.mean = {c: (float(subset[c].mean()) if len(subset) else 0.0) for c in feature_cols}
    return root


def soft_membership(row, leaves, temperature=0.04):
    point = np.array([float(row["greed_index"]), float(row["fear_index"])])
    centers = np.array([[(l.bounds["greed_index"][0] + l.bounds["greed_index"][1]) / 2.0,
                          (l.bounds["fear_index"][0] + l.bounds["fear_index"][1]) / 2.0]
                         for l in leaves])
    distances = np.sum((centers - point[None, :]) ** 2, axis=1)
    logits = -distances / max(temperature, 1e-8)
    logits = logits - logits.max()
    w = np.exp(logits)
    return w / w.sum()


def add_memberships(frame, leaves, membership_columns):
    matrix = np.vstack([soft_membership(row, leaves) for _, row in frame.iterrows()])
    df_m = pd.DataFrame(matrix, columns=membership_columns, index=frame.index)
    return pd.concat([frame.reset_index(drop=True), df_m.reset_index(drop=True)], axis=1)


def time_split(frame, validation_fraction, test_fraction):
    n = len(frame)
    test_start = int(n * (1.0 - test_fraction))
    validation_start = int(test_start * (1.0 - validation_fraction))
    return (
        frame.iloc[:validation_start].copy(),
        frame.iloc[validation_start:test_start].copy(),
        frame.iloc[test_start:].copy(),
    )


## Discrete state assignment

`state_t` is the index of the (unique) leaf whose bounds contain `(greed_t, fear_t)`. The target is `state_{t+HORIZON}`. Rows where the future state can't be computed (last `HORIZON` rows) are dropped.


In [3]:
def assign_leaf_index(row, leaves):
    for i, leaf in enumerate(leaves):
        if in_bounds(row, leaf.bounds):
            return i
    return -1


def add_states(frame, leaves, horizon):
    frame = frame.copy()
    frame["state_t"] = frame.apply(lambda r: assign_leaf_index(r, leaves), axis=1)
    frame["state_t_future"] = frame["state_t"].shift(-horizon)
    # Drop rows that fell outside any leaf (shouldn't happen if hierarchy covers [0,1]^2) or have no future
    return frame.dropna(subset=["state_t_future"]).reset_index(drop=True).astype({"state_t": int, "state_t_future": int})


## Models

Four predictors + two trivial baselines (random, marginal). The `empirical_markov_predict` is the count-based kernel from NB02; everything else uses RF with validation-tuned hyperparameters.


In [4]:
def estimate_transition_matrix(state_t, state_t_future, K):
    M = np.zeros((K, K))
    for s, u in zip(state_t, state_t_future):
        M[int(s), int(u)] += 1
    row_sums = M.sum(axis=1, keepdims=True)
    return M / np.maximum(row_sums, 1.0)


def empirical_markov_predict(state_t, transition_matrix):
    # Predict argmax_u P(z_{t+h}=u | z_t=s) for each test row
    pred_table = transition_matrix.argmax(axis=1)  # shape (K,)
    return np.array([pred_table[int(s)] for s in state_t])


def marginal_majority_predict(state_t_train, n_rows):
    counts = pd.Series(state_t_train).value_counts()
    majority = int(counts.idxmax())
    return np.full(n_rows, majority, dtype=int)


def random_predict(n_rows, K, rng):
    return rng.integers(0, K, size=n_rows)


def _rf_cls(params, K=None):
    return RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS, random_state=SEED, n_jobs=-1,
        class_weight="balanced_subsample",
        **params,
    )


def select_rf_cls_params(x_train, y_train, x_val, y_val):
    best_params, best_score = None, -np.inf
    for params in RF_GRID:
        m = _rf_cls(params); m.fit(x_train, y_train)
        s = f1_score(y_val, m.predict(x_val), average="macro")
        if s > best_score:
            best_params, best_score = params, s
    return best_params, best_score


def fit_rf_classifier(feature_columns, train_df, val_df, test_df):
    x_train = train_df[feature_columns]; y_train = train_df["state_t_future"].astype(int)
    x_val = val_df[feature_columns]; y_val = val_df["state_t_future"].astype(int)
    x_test = test_df[feature_columns]; y_test = test_df["state_t_future"].astype(int)
    best_params, val_macro_f1 = select_rf_cls_params(x_train, y_train, x_val, y_val)
    x_trainval = pd.concat([x_train, x_val]); y_trainval = pd.concat([y_train, y_val])
    m = _rf_cls(best_params); m.fit(x_trainval, y_trainval)
    pred = m.predict(x_test)
    return pred, m, best_params, float(val_macro_f1)


## Per-ticker pipeline + cross-ticker run

For each ticker we build the hierarchy from training data, assign states, run all six predictors, and score on the test slice. Transition-only accuracy is computed over the subset of test rows where the state actually changed (i.e., `state_t_future != state_t`) — this is the harder, more interesting metric because persistence gets it wrong by construction.


In [5]:
def score_predictions(y_true, y_pred, state_t):
    transition_mask = (y_true != state_t)
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "transition_count": int(transition_mask.sum()),
        "transition_accuracy": float(accuracy_score(y_true[transition_mask], y_pred[transition_mask])) if transition_mask.any() else float("nan"),
    }
    return out


def run_ticker(ticker, rng):
    df = load_yfinance_ohlcv(ticker, START_DATE, END_DATE)
    feat = make_real_market_features(df, HORIZON)
    train_df, val_df, test_df = time_split(feat, VALIDATION_FRACTION, TEST_FRACTION)

    root = build_hierarchy(train_df, max_depth=MAX_HIERARCHY_DEPTH, min_count=MIN_LEAF_COUNT)
    leaves = collect_leaves(root)
    K = len(leaves)
    membership_columns = [f"leaf_{leaf.node_id}" for leaf in leaves]

    # Assign states (uses leaf bounds, which were built from train_df)
    train_s = add_states(train_df, leaves, HORIZON)
    val_s = add_states(val_df, leaves, HORIZON)
    test_s = add_states(test_df, leaves, HORIZON)

    # Add soft memberships
    train_m = add_memberships(train_s, leaves, membership_columns)
    val_m = add_memberships(val_s, leaves, membership_columns)
    test_m = add_memberships(test_s, leaves, membership_columns)

    y_test = test_m["state_t_future"].astype(int).to_numpy()
    state_t_test = test_m["state_t"].astype(int).to_numpy()

    # Baselines
    pred_random = random_predict(len(test_m), K, rng)
    pred_marginal = marginal_majority_predict(train_m["state_t_future"].astype(int), len(test_m))
    pred_persistence = test_m["state_t"].astype(int).to_numpy()
    T = estimate_transition_matrix(
        train_m["state_t"].astype(int).to_numpy(),
        train_m["state_t_future"].astype(int).to_numpy(),
        K,
    )
    pred_markov = empirical_markov_predict(state_t_test, T)

    # RF models
    feat_only_features = BASELINE_FEATURES
    joint_features = BASELINE_FEATURES + membership_columns

    pred_feat, m_feat, p_feat, val_f1_feat = fit_rf_classifier(feat_only_features, train_m, val_m, test_m)
    pred_joint, m_joint, p_joint, val_f1_joint = fit_rf_classifier(joint_features, train_m, val_m, test_m)

    rows = []
    for name, pred in [
        ("random", pred_random),
        ("marginal", pred_marginal),
        ("persistence", pred_persistence),
        ("empirical_markov", pred_markov),
        ("rf_features_only", pred_feat),
        ("rf_joint", pred_joint),
    ]:
        s = score_predictions(y_test, pred, state_t_test)
        s["model"] = name
        s["ticker"] = ticker
        s["K"] = K
        s["n_train"] = len(train_m); s["n_val"] = len(val_m); s["n_test"] = len(test_m)
        rows.append(s)
    info = {
        "transition_matrix": T,
        "rf_feat_params": p_feat,
        "rf_joint_params": p_joint,
        "val_f1_feat": val_f1_feat,
        "val_f1_joint": val_f1_joint,
        "leaf_centers": [(l.bounds["greed_index"], l.bounds["fear_index"]) for l in leaves],
    }
    return pd.DataFrame(rows), info, (m_feat, m_joint, feat_only_features, joint_features, test_m)


rng = np.random.default_rng(SEED)
all_results = []
all_info = {}
all_models = {}
for tk in TICKERS:
    print(f"Running {tk}...")
    res_df, info, models = run_ticker(tk, rng)
    all_results.append(res_df)
    all_info[tk] = info
    all_models[tk] = models

agg = pd.concat(all_results, ignore_index=True)
agg[["ticker", "model", "K", "n_train", "n_val", "n_test",
     "accuracy", "macro_f1", "transition_count", "transition_accuracy"]]


Running SPY...


Running NVDA...


Running BTC-USD...


,ticker,model,K,n_train,n_val,n_test,accuracy,macro_f1,transition_count,transition_accuracy
0,SPY,random,4,2754,483,807,0.245353,0.245395,522,0.250958
1,SPY,marginal,4,2754,483,807,0.252788,0.100890,522,0.281609
2,SPY,persistence,4,2754,483,807,0.353160,0.352704,522,0.000000
3,SPY,empirical_markov,4,2754,483,807,0.360595,0.300351,522,0.214559
4,SPY,rf_features_only,4,2754,483,807,0.463445,0.445156,522,0.383142
5,SPY,rf_joint,4,2754,483,807,0.486989,0.473335,522,0.394636
6,NVDA,random,4,2754,483,807,0.232962,0.232149,491,0.246436
7,NVDA,marginal,4,2754,483,807,0.276332,0.108252,491,0.250509
8,NVDA,persistence,4,2754,483,807,0.391574,0.386774,491,0.000000
9,NVDA,empirical_markov,4,2754,483,807,0.391574,0.386774,491,0.000000


## Cross-ticker summary

Pivot the test-set metrics by `ticker × model`. The columns to watch:

- `accuracy` vs `random=1/K`: any positive gap.
- `accuracy` vs `persistence`: this is the real bar.
- `transition_accuracy`: persistence is *zero* by construction on transitions; the empirical Markov kernel and the two RFs are the only models that can score above 0 here.


In [6]:
metrics_view = ["accuracy", "macro_f1", "transition_accuracy"]
pivot = agg.pivot_table(index="model", columns="ticker", values=metrics_view)
pivot = pivot.reorder_levels([1, 0], axis=1).sort_index(axis=1)
pivot


ticker             BTC-USD                                    NVDA            \
                  accuracy  macro_f1 transition_accuracy  accuracy  macro_f1   
model                                                                          
empirical_markov  0.421053  0.402948            0.000000  0.391574  0.386774   
marginal          0.230861  0.093780            0.223140  0.276332  0.108252   
persistence       0.421053  0.402948            0.000000  0.391574  0.386774   
random            0.251196  0.245734            0.239669  0.232962  0.232149   
rf_features_only  0.489234  0.455661            0.314050  0.462206  0.448400   
rf_joint          0.495215  0.462127            0.291322  0.462206  0.441213   

ticker                                     SPY                                
                 transition_accuracy  accuracy  macro_f1 transition_accuracy  
model                                                                         
empirical_markov            0.000000  0.360595  0.300351            0.214559  
marginal                    0.250509  0.252788  0.100890            0.281609  
persistence                 0.000000  0.353160  0.352704            0.000000  
random                      0.246436  0.245353  0.245395            0.250958  
rf_features_only            0.323829  0.463445  0.445156            0.383142  
rf_joint                    0.297352  0.486989  0.473335            0.394636

In [7]:
for tk in TICKERS:
    info = all_info[tk]
    print(f"\n=== {tk} — empirical transition matrix P(z_{{t+5}} | z_t) ===")
    T = info["transition_matrix"]
    df_T = pd.DataFrame(T, index=[f"from_{i}" for i in range(T.shape[0])], columns=[f"to_{j}" for j in range(T.shape[1])])
    print(df_T.round(3).to_string())
    print(f"rf_features_only best params: {info['rf_feat_params']} (val macro-F1 = {info['val_f1_feat']:.4f})")
    print(f"rf_joint best params:         {info['rf_joint_params']} (val macro-F1 = {info['val_f1_joint']:.4f})")



=== SPY — empirical transition matrix P(z_{t+5} | z_t) ===
         to_0   to_1   to_2   to_3
from_0  0.436  0.245  0.160  0.159
from_1  0.321  0.276  0.212  0.190
from_2  0.178  0.296  0.284  0.242
from_3  0.067  0.186  0.342  0.406
rf_features_only best params: {'max_depth': 8, 'min_samples_leaf': 10} (val macro-F1 = 0.4568)
rf_joint best params:         {'max_depth': 8, 'min_samples_leaf': 10} (val macro-F1 = 0.4499)

=== NVDA — empirical transition matrix P(z_{t+5} | z_t) ===
         to_0   to_1   to_2   to_3
from_0  0.373  0.225  0.222  0.180
from_1  0.190  0.453  0.185  0.172
from_2  0.265  0.191  0.281  0.262
from_3  0.172  0.133  0.310  0.384
rf_features_only best params: {'max_depth': 8, 'min_samples_leaf': 10} (val macro-F1 = 0.5244)
rf_joint best params:         {'max_depth': 6, 'min_samples_leaf': 40} (val macro-F1 = 0.5315)

=== BTC-USD — empirical transition matrix P(z_{t+5} | z_t) ===
         to_0   to_1   to_2   to_3
from_0  0.473  0.185  0.211  0.130
from_1  0.168  

## Permutation importance on the joint RF (per ticker)

If the soft-membership columns (`leaf_*`) contribute genuine signal in the Markov-kernel setting, we should see them with positive permutation importance on the test set. If they're at zero/negative — i.e., the joint RF is also just using baseline features — then the formalism doesn't add information beyond what's already in returns/volatility.


In [8]:
def perm_table(model, x_test, y_test, feature_columns, scoring, n_repeats=10):
    pi = permutation_importance(
        model, x_test, y_test,
        n_repeats=n_repeats, random_state=SEED, n_jobs=-1,
        scoring=scoring,
    )
    return pd.DataFrame({
        "feature": feature_columns,
        "perm_importance_mean": pi.importances_mean,
        "perm_importance_std": pi.importances_std,
    }).sort_values("perm_importance_mean", ascending=False).reset_index(drop=True)


perm_by_ticker = {}
for tk in TICKERS:
    m_feat, m_joint, feat_only, joint, test_m = all_models[tk]
    perm = perm_table(m_joint, test_m[joint], test_m["state_t_future"].astype(int), joint, scoring="f1_macro")
    perm_by_ticker[tk] = perm
    print(f"=== {tk} (joint RF, scoring=macro F1) ===")
    print(perm.to_string(index=False))
    print()


=== SPY (joint RF, scoring=macro F1) ===
       feature  perm_importance_mean  perm_importance_std
         ret_5              0.105809             0.010175
price_to_ma_50              0.020587             0.006806
price_to_ma_20              0.019580             0.006405
        ret_10              0.016912             0.007134
        ret_20              0.016576             0.005482
        vol_10              0.012007             0.005784
   leaf_rootRL              0.010336             0.003466
price_to_ma_10              0.010245             0.007010
   drawdown_50              0.007196             0.005200
   volume_z_20              0.006311             0.005576
   drawdown_20              0.004810             0.003737
         vol_5              0.003597             0.004836
   leaf_rootLR              0.003112             0.002408
   leaf_rootRR              0.002695             0.008769
         ret_2              0.000882             0.004222
        vol_20             -0.0

=== NVDA (joint RF, scoring=macro F1) ===
       feature  perm_importance_mean  perm_importance_std
         ret_5              0.072800             0.013664
price_to_ma_50              0.015706             0.009331
price_to_ma_20              0.005962             0.003812
        ret_10              0.005572             0.002212
   drawdown_50              0.004508             0.007046
     range_pct              0.002457             0.002065
   leaf_rootRL              0.001432             0.004916
price_to_ma_10              0.001310             0.006364
        ret_20              0.000131             0.004334
        vol_10             -0.000491             0.001769
   volume_z_20             -0.000964             0.002094
   drawdown_20             -0.001191             0.005075
         vol_5             -0.002556             0.002740
   leaf_rootLR             -0.002569             0.005221
open_close_pct             -0.003539             0.001512
         ret_1             -0.

=== BTC-USD (joint RF, scoring=macro F1) ===
       feature  perm_importance_mean  perm_importance_std
         ret_5              0.060338             0.009292
price_to_ma_50              0.014582             0.006308
price_to_ma_10              0.005775             0.004979
        ret_20              0.004325             0.003796
   leaf_rootRL              0.001758             0.001966
   drawdown_50              0.001674             0.004107
open_close_pct              0.000700             0.002948
        ret_10              0.000672             0.004967
         ret_1              0.000407             0.003031
   volume_z_20              0.000393             0.001285
        vol_20              0.000326             0.004307
         ret_2              0.000267             0.004972
     range_pct             -0.000055             0.002827
         vol_5             -0.000395             0.002150
        vol_10             -0.000857             0.003262
   leaf_rootLR             

## Append to EXPERIMENTS.md


In [9]:
def append_to_journal(agg_df, all_info, journal_path=JOURNAL_PATH):
    header_needed = not os.path.exists(journal_path)
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    config = {
        "notebook": "06_real_data_markov_kernel.ipynb",
        "tickers": TICKERS,
        "start_date": START_DATE,
        "horizon": HORIZON,
        "max_hierarchy_depth": MAX_HIERARCHY_DEPTH,
        "min_leaf_count": MIN_LEAF_COUNT,
        "rf_n_estimators": RF_N_ESTIMATORS,
        "rf_grid": RF_GRID,
        "seed": SEED,
    }

    cols = ["ticker", "model", "K", "n_train", "n_val", "n_test",
            "accuracy", "macro_f1", "transition_count", "transition_accuracy"]
    t = agg_df[cols].copy()
    for c in ("accuracy", "macro_f1", "transition_accuracy"):
        t[c] = t[c].map(lambda v: f"{v:.4f}" if pd.notna(v) else "nan")

    parts = []
    if header_needed:
        parts.append("# Experiments journal\n\nAppend-only log of model-comparison runs.\n")
    parts.append(f"\n---\n")
    parts.append(f"\n## {timestamp} — NB06 Markov-kernel on real data (K={2**MAX_HIERARCHY_DEPTH}, horizon={HORIZON})\n")
    parts.append(f"\n**Config**\n\n```json\n{json.dumps(config, indent=2)}\n```\n")
    parts.append(f"\n**Per-ticker test-set results**\n\n")
    parts.append(t.to_markdown(index=False))
    parts.append("\n\n**Empirical transition matrices (training data)**\n\n")
    for tk in TICKERS:
        T = all_info[tk]["transition_matrix"]
        parts.append(f"\n`{tk}`\n\n```\n{pd.DataFrame(T).round(3).to_string()}\n```\n")
    parts.append("\n**RF hyperparameter choices (val macro-F1)**\n\n")
    rows = []
    for tk in TICKERS:
        info = all_info[tk]
        rows.append({"ticker": tk,
                     "rf_features_only": str(info["rf_feat_params"]),
                     "val_f1_features": f"{info['val_f1_feat']:.4f}",
                     "rf_joint": str(info["rf_joint_params"]),
                     "val_f1_joint": f"{info['val_f1_joint']:.4f}"})
    parts.append(pd.DataFrame(rows).to_markdown(index=False))
    parts.append("\n")
    text = "".join(parts)
    with open(journal_path, "a") as f:
        f.write(text)
    print(f"Appended entry to {journal_path}")
    return text


journal_text = append_to_journal(agg, all_info)
print()
print(journal_text)


Appended entry to EXPERIMENTS.md


---

## 2026-05-12 15:19:06 — NB06 Markov-kernel on real data (K=4, horizon=5)

**Config**

```json
{
  "notebook": "06_real_data_markov_kernel.ipynb",
  "tickers": [
    "SPY",
    "NVDA",
    "BTC-USD"
  ],
  "start_date": "2010-01-01",
  "horizon": 5,
  "max_hierarchy_depth": 2,
  "min_leaf_count": 200,
  "rf_n_estimators": 300,
  "rf_grid": [
    {
      "max_depth": 4,
      "min_samples_leaf": 10
    },
    {
      "max_depth": 4,
      "min_samples_leaf": 20
    },
    {
      "max_depth": 4,
      "min_samples_leaf": 40
    },
    {
      "max_depth": 6,
      "min_samples_leaf": 10
    },
    {
      "max_depth": 6,
      "min_samples_leaf": 20
    },
    {
      "max_depth": 6,
      "min_samples_leaf": 40
    },
    {
      "max_depth": 8,
      "min_samples_leaf": 10
    },
    {
      "max_depth": 8,
      "min_samples_leaf": 20
    },
    {
      "max_depth": 8,
      "min_samples_leaf": 40
    }
  ],
  "seed": 42
}
```

**Per-ticker tes

## What to look for

1. **Does any model beat persistence on accuracy?** If yes, there's structure beyond trivial autocorrelation.
2. **Transition accuracy**: persistence is 0 by construction. The empirical Markov kernel makes a deterministic prediction per `z_t`, so its transition accuracy reflects whether the *modal* next state given `z_t` is ever a state ≠ `z_t`. Both RFs can in principle score higher.
3. **Does `rf_joint` beat `rf_features_only`?** If yes, the soft-membership representation adds information beyond plain price/volume features. If they're tied, the baseline features alone capture the relevant dynamics.
4. **Permutation importance on `rf_joint`**: are `leaf_*` columns positive on held-out data, or is the joint RF also just leaning on `ret_20` and `price_to_ma_50`?
